In [ ]:

!pip --version

pip 25.1 from C:\Users\jeeva\anaconda3\Lib\site-packages\pip (python 3.13)



In [2]:
!pip install -U "transformers>=4.41" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4"



In [3]:
# ==== Clean end-to-end cell (4 spaces only, no tabs) ====
import os
import random
import numpy as np
import torch

from datasets import load_dataset, interleave_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

# ---- assumes you already defined these earlier; redefine here for safety ----
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
SRC_LANG = "en_XX"
HF_DATASETS = {
    "en-es": {"path": "opus100", "config": "en-es"},
    "en-pt": {"path": "opus100", "config": "en-pt"},
}
MAX_SAMPLES_PER_LANG = None
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")

def set_seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed_all(SEED)

# 1) Tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
tokenizer.src_lang = SRC_LANG

# 2) Optional simple glossary (hybrid hook)
GLOSSARY = {
    "polymerase chain reaction": {"es": "reacción en cadena de la polimerasa", "pt": "reação em cadeia da polimerase"},
    "confidence interval": {"es": "intervalo de confianza", "pt": "intervalo de confiança"},
}
def apply_glossary(text: str, tgt_lang_code: str) -> str:
    tgt_key = "es" if tgt_lang_code == "es_XX" else ("pt" if tgt_lang_code == "pt_XX" else None)
    if not tgt_key:
        return text
    out = text
    for en_term, mapping in GLOSSARY.items():
        if tgt_key in mapping:
            out = out.replace(en_term, mapping[tgt_key])
    return out

# 3) Robust dataset loader (handles 'translation' schema or source/target)
def _pair_from_key(key: str):
    src, tgt = key.split("-")  # "en-es" -> ("en","es")
    return src, tgt

def _map_example(ex, tgt_code: str, src_lang_key: str, tgt_lang_key: str):
    if "translation" in ex:
        src = ex["translation"][src_lang_key]
        tgt = ex["translation"][tgt_lang_key]
    elif "source" in ex and "target" in ex:
        src = ex["source"]
        tgt = ex["target"]
    elif src_lang_key in ex and tgt_lang_key in ex:
        src = ex[src_lang_key]
        tgt = ex[tgt_lang_key]
    else:
        raise KeyError(
            f"Unexpected schema. Keys present: {list(ex.keys())}. "
            f"Expected 'translation' or ('source','target') or ('{src_lang_key}','{tgt_lang_key}')."
        )
    return {"src_text": src, "tgt_text": tgt, "tgt_lang": tgt_code}

def load_mt_split(split: str = "train"):
    parts = []
    for key, spec in HF_DATASETS.items():
        ds = load_dataset(spec["path"], spec["config"], split=split)
        if MAX_SAMPLES_PER_LANG:
            ds = ds.select(range(min(MAX_SAMPLES_PER_LANG, len(ds))))
        tgt_code = "es_XX" if "en-es" in key else "pt_XX"
        src_lang_key, tgt_lang_key = _pair_from_key(key)
        ds = ds.map(
            lambda ex: _map_example(ex, tgt_code, src_lang_key, tgt_lang_key),
            remove_columns=ds.column_names
        )
        parts.append(ds)
    return interleave_datasets(parts, seed=SEED)

train_raw = load_mt_split("train")
valid_raw = load_mt_split("validation")
print(f"Train size: {len(train_raw):,} | Valid size: {len(valid_raw):,}")
print("Sample:", train_raw[0])

# 4) Preprocess/tokenize
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

def make_preprocess_fn(tgt_code: str):
    def _fn(batch):
        # glossary before tokenization
        glossed_targets = [apply_glossary(t, tgt_code) for t in batch["tgt_text"]]

        # set languages for this whole batch
        tokenizer.src_lang = SRC_LANG
        tokenizer.tgt_lang = tgt_code  # REQUIRED for mBART50

        # tokenize inputs
        model_inputs = tokenizer(
            batch["src_text"],
            max_length=MAX_SOURCE_LEN,
            truncation=True,
        )

        # tokenize labels for this fixed target language
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                glossed_targets,
                max_length=MAX_TARGET_LEN,
                truncation=True,
            )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return _fn

# Split raw datasets by target language
train_raw_es = train_raw.filter(lambda ex: ex["tgt_lang"] == "es_XX")
train_raw_pt = train_raw.filter(lambda ex: ex["tgt_lang"] == "pt_XX")
valid_raw_es = valid_raw.filter(lambda ex: ex["tgt_lang"] == "es_XX")
valid_raw_pt = valid_raw.filter(lambda ex: ex["tgt_lang"] == "pt_XX")

# Map with language-specific preprocessors
pre_es = make_preprocess_fn("es_XX")
pre_pt = make_preprocess_fn("pt_XX")

train_ds_es = train_raw_es.map(pre_es, batched=True, remove_columns=train_raw_es.column_names)
train_ds_pt = train_raw_pt.map(pre_pt, batched=True, remove_columns=train_raw_pt.column_names)
valid_ds_es = valid_raw_es.map(pre_es, batched=True, remove_columns=valid_raw_es.column_names)
valid_ds_pt = valid_raw_pt.map(pre_pt, batched=True, remove_columns=valid_raw_pt.column_names)

# Interleave AFTER tokenization so batches are homogeneous wrt tgt_lang
from datasets import interleave_datasets
train_ds = interleave_datasets([train_ds_es, train_ds_pt], seed=SEED)
valid_ds = interleave_datasets([valid_ds_es, valid_ds_pt], seed=SEED)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

🔥 Using device: cuda


c:\Users\jeeva\anaconda3\Lib\site-packages\torch\cuda\__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5070 Laptop GPU which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (5.0) - (9.0)
    
  warnings.warn(
c:\Users\jeeva\anaconda3\Lib\site-packages\torch\cuda\__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 13.0 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
c:\Users\jeeva\anaconda3\Lib\site-packages\torch\cuda\__init__.py:326: UserWarning: 
NVIDIA GeForce RTX 5070 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070 Laptop GPU GPU with PyTorch, please check the instructio

Train size: 2,000,000 | Valid size: 4,000
Sample: {'src_text': "It was the asbestos in here, that's what did it!", 'tgt_text': 'Fueron los asbestos aquí. ¡Eso es lo que ocurrió!', 'tgt_lang': 'es_XX'}


Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

c:\Users\jeeva\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
!pip install transformers sentencepiece datasets sacrebleu evaluate accelerate 

In [5]:
# ==== Clean end-to-end cell (4 spaces only, no tabs) ====
import os
import random
import numpy as np
import torch

from datasets import load_dataset, interleave_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

# ---- assumes you already defined these earlier; redefine here for safety ----
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
SRC_LANG = "en_XX"
HF_DATASETS = {
    "en-es": {"path": "opus100", "config": "en-es"},
    "en-pt": {"path": "opus100", "config": "en-pt"},
}
MAX_SAMPLES_PER_LANG = None
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")

def set_seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed_all(SEED)

# 1) Tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
tokenizer.src_lang = SRC_LANG

# 2) Optional simple glossary (hybrid hook)
GLOSSARY = {
    "polymerase chain reaction": {"es": "reacción en cadena de la polimerasa", "pt": "reação em cadeia da polimerase"},
    "confidence interval": {"es": "intervalo de confianza", "pt": "intervalo de confiança"},
}
def apply_glossary(text: str, tgt_lang_code: str) -> str:
    tgt_key = "es" if tgt_lang_code == "es_XX" else ("pt" if tgt_lang_code == "pt_XX" else None)
    if not tgt_key:
        return text
    out = text
    for en_term, mapping in GLOSSARY.items():
        if tgt_key in mapping:
            out = out.replace(en_term, mapping[tgt_key])
    return out

# 3) Robust dataset loader (handles 'translation' schema or source/target)
def _pair_from_key(key: str):
    src, tgt = key.split("-")  # "en-es" -> ("en","es")
    return src, tgt

def _map_example(ex, tgt_code: str, src_lang_key: str, tgt_lang_key: str):
    if "translation" in ex:
        src = ex["translation"][src_lang_key]
        tgt = ex["translation"][tgt_lang_key]
    elif "source" in ex and "target" in ex:
        src = ex["source"]
        tgt = ex["target"]
    elif src_lang_key in ex and tgt_lang_key in ex:
        src = ex[src_lang_key]
        tgt = ex[tgt_lang_key]
    else:
        raise KeyError(
            f"Unexpected schema. Keys present: {list(ex.keys())}. "
            f"Expected 'translation' or ('source','target') or ('{src_lang_key}','{tgt_lang_key}')."
        )
    return {"src_text": src, "tgt_text": tgt, "tgt_lang": tgt_code}

def load_mt_split(split: str = "train"):
    parts = []
    for key, spec in HF_DATASETS.items():
        ds = load_dataset(spec["path"], spec["config"], split=split)
        if MAX_SAMPLES_PER_LANG:
            ds = ds.select(range(min(MAX_SAMPLES_PER_LANG, len(ds))))
        tgt_code = "es_XX" if "en-es" in key else "pt_XX"
        src_lang_key, tgt_lang_key = _pair_from_key(key)
        ds = ds.map(
            lambda ex: _map_example(ex, tgt_code, src_lang_key, tgt_lang_key),
            remove_columns=ds.column_names
        )
        parts.append(ds)
    return interleave_datasets(parts, seed=SEED)

train_raw = load_mt_split("train")
valid_raw = load_mt_split("validation")
print(f"Train size: {len(train_raw):,} | Valid size: {len(valid_raw):,}")
print("Sample:", train_raw[0])

# 4) Preprocess/tokenize
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

def make_preprocess_fn(tgt_code: str):
    def _fn(batch):
        # glossary before tokenization
        glossed_targets = [apply_glossary(t, tgt_code) for t in batch["tgt_text"]]

        # set languages for this whole batch
        tokenizer.src_lang = SRC_LANG
        tokenizer.tgt_lang = tgt_code  # REQUIRED for mBART50

        # tokenize inputs
        model_inputs = tokenizer(
            batch["src_text"],
            max_length=MAX_SOURCE_LEN,
            truncation=True,
        )

        # tokenize labels for this fixed target language
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                glossed_targets,
                max_length=MAX_TARGET_LEN,
                truncation=True,
            )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return _fn

# Split raw datasets by target language
train_raw_es = train_raw.filter(lambda ex: ex["tgt_lang"] == "es_XX")
train_raw_pt = train_raw.filter(lambda ex: ex["tgt_lang"] == "pt_XX")
valid_raw_es = valid_raw.filter(lambda ex: ex["tgt_lang"] == "es_XX")
valid_raw_pt = valid_raw.filter(lambda ex: ex["tgt_lang"] == "pt_XX")

# Map with language-specific preprocessors
pre_es = make_preprocess_fn("es_XX")
pre_pt = make_preprocess_fn("pt_XX")

train_ds_es = train_raw_es.map(pre_es, batched=True, remove_columns=train_raw_es.column_names)
train_ds_pt = train_raw_pt.map(pre_pt, batched=True, remove_columns=train_raw_pt.column_names)
valid_ds_es = valid_raw_es.map(pre_es, batched=True, remove_columns=valid_raw_es.column_names)
valid_ds_pt = valid_raw_pt.map(pre_pt, batched=True, remove_columns=valid_raw_pt.column_names)

# Interleave AFTER tokenization so batches are homogeneous wrt tgt_lang
from datasets import interleave_datasets
train_ds = interleave_datasets([train_ds_es, train_ds_pt], seed=SEED)
valid_ds = interleave_datasets([valid_ds_es, valid_ds_pt], seed=SEED)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

🔥 Using device: cuda
Train size: 2,000,000 | Valid size: 4,000
Sample: {'src_text': "It was the asbestos in here, that's what did it!", 'tgt_text': 'Fueron los asbestos aquí. ¡Eso es lo que ocurrió!', 'tgt_lang': 'es_XX'}


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
# =========================
# Cell — Quick BLEU evaluate + samples
# =========================
import os
import evaluate
from tqdm import tqdm

# We'll reuse the raw, language-tagged validation sets from earlier:
# valid_raw_es, valid_raw_pt each have fields: src_text, tgt_text, tgt_lang

# Limit for speed while debugging (set to None for full split)
EVAL_SAMPLES = 300
if EVAL_SAMPLES is not None:
    valid_raw_es_eval = valid_raw_es.select(range(min(EVAL_SAMPLES, len(valid_raw_es))))
    valid_raw_pt_eval = valid_raw_pt.select(range(min(EVAL_SAMPLES, len(valid_raw_pt))))
else:
    valid_raw_es_eval = valid_raw_es
    valid_raw_pt_eval = valid_raw_pt

bleu = evaluate.load("sacrebleu")

def generate_batch(texts, tgt_lang_code, max_new_tokens=128, batch_size=16):
    """
    Generate translations for English `texts` into `tgt_lang_code` (e.g., 'es_XX', 'pt_XX').
    """
    model.eval()
    tokenizer.src_lang = SRC_LANG

    prev_forced = getattr(model.config, "forced_bos_token_id", None)
    model.config.forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang_code]

    outputs = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,              # <-- key fix
            max_length=MAX_SOURCE_LEN,
            pad_to_multiple_of=8 if torch.cuda.is_available() else None,  # nicer for fp16 on GPU
        ).to(device)

        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                num_beams=4
            )

        outputs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))

    model.config.forced_bos_token_id = prev_forced
    return outputs


In [7]:
from transformers import EarlyStoppingCallback
import evaluate

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # decode preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # replace -100 in labels before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # sacrebleu expects list of refs
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["score"]}

args = TrainingArguments(
    output_dir="./mt_bilingual_en_es_pt/checkpoints",
    evaluation_strategy="steps",
    eval_steps=1000,                  # tune for your speed
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    logging_steps=100,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,               # ↑ from 1
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,            # <-- add eval
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # <-- add metrics
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'